# P18 · U4 + U5: bias and memorisation tests, filter check, diffusion mascot
Runtime ▸ **T4 GPU**, then Run all (about 10 min). Cells run in order: tests first, mascot last.

## Setup

In [ ]:
import os, sys, re, json, time, random, zipfile, shutil, glob, subprocess
from collections import Counter
import torch, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
dev = "cuda" if torch.cuda.is_available() else "cpu"
for d in ("src", "out", "data"): os.makedirs(d, exist_ok=True)
if not os.path.exists("src/model.py") or not os.path.exists("out/mini_gpt.pt"):
    from google.colab import files
    files.upload()                       # choose p18_repo.zip
    zipfile.ZipFile("p18_repo.zip").extractall(".")
    for f in glob.glob("p18repo/src/*.py"): shutil.copy(f, "src/")
    for f in glob.glob("p18repo/out/*"): shutil.copy(f, "out/")
if not os.path.exists("data/train.txt"):
    print("data/ missing: rebuilding the corpus (shares are computed on the fresh train split).")
    subprocess.run([sys.executable, "src/data.py"], check=True)
sys.path.insert(0, "src")
from sample import load_model, generate
from safety import is_safe, filter_names
tr = [l for l in open("data/train.txt").read().split("\n") if l]
known = set(tr)
model, stoi, itos = load_model("out/mini_gpt.pt", dev)
print("corpus names:", len(tr), "| device:", dev)

## Test 1: bias in regional suffix families

In [ ]:
# TEST 1: suffix-family bias. Do generated names keep the corpus mix of regional naming patterns,
# or do they collapse towards the most common patterns?  (Suffix families only approximate regions.)
GROUPS = {
  "North / Hindi belt": ["pur", "pura", "ganj", "garh", "khera", "nagar", "abad", "wala", "pore"],
  "Tamil":              ["patti", "palayam", "kudi", "kulam", "kottai"],
  "Telugu / Kannada":   ["halli", "palli", "palle", "peta", "konda", "gudem", "pally", "kere"],
  "Bengal / Odisha":    ["pukur", "danga", "para", "pada", "bari"],
  "Marathi / Gujarat":  ["wadi", "wada", "gaon", "pada"],
  "Kerala":             ["kunnu", "mangalam", "kkal", "kkara"],
}
PAIRS = sorted([(s, g) for g, ss in GROUPS.items() for s in ss], key=lambda x: -len(x[0]))
def family(name):
    last = name.split()[-1]
    for s, g in PAIRS:
        if last.endswith(s) and len(last) > len(s): return g
    return "Other"
def shares(names):
    c = Counter(family(n) for n in names); t = len(names)
    return {g: 100 * c.get(g, 0) / t for g in list(GROUPS) + ["Other"]}
def gen(T, n=5000, seed=0):
    return [x.lower() for x in generate(model, stoi, itos, n=n, temperature=T, seed=seed) if x]

samples = {T: gen(T) for T in (0.5, 0.8, 1.2)}
tab = pd.DataFrame({"corpus": shares(tr), **{f"T={T}": shares(v) for T, v in samples.items()}})
tvd = {c: 0.5 * (tab[c] - tab["corpus"]).abs().sum() for c in tab.columns if c != "corpus"}
print(tab.round(2).to_string()); print("\nTotal variation distance from corpus (0 = same mix, 100 = disjoint):")
print({k: round(v, 2) for k, v in tvd.items()})
tab.round(3).to_csv("out/bias_suffix_shares.csv")
ax = tab[["corpus", "T=0.5", "T=0.8", "T=1.2"]].plot(kind="bar", figsize=(9, 4.5))
ax.set_ylabel("% of names"); ax.set_title("Regional suffix families: corpus vs generated"); plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.savefig("out/bias_chart.png", dpi=150)

## Test 2: memorisation (copy rate)

In [ ]:
# TEST 2: memorisation (the "hallucination" analogue here): how often is a generated name an exact copy of a real one?
rows = {}
for T, v in samples.items():
    rows[f"T={T}"] = {"copy_rate_%": 100 * sum(n in known for n in v) / len(v),
                      "unique_%": 100 * len(set(v)) / len(v)}
print(pd.DataFrame(rows).T.round(2).to_string())
print("\nMost repeated names at T=0.5:", Counter(samples[0.5]).most_common(8))
print("Examples of copied real names at T=0.8:", [n for n in samples[0.8] if n in known][:8])
pd.DataFrame(rows).T.round(3).to_csv("out/memorisation.csv")

## Test 3: safety filter false positives

In [ ]:
# TEST 3: safety filter. Measure its false positives on REAL place names and what it blocks in generated output.
real_blocked = [n for n in tr if not is_safe(n)]
gen_blocked = [n for n in samples[0.8] if not is_safe(n)]
print(f"real corpus names blocked: {len(real_blocked)} of {len(tr)} ({100*len(real_blocked)/len(tr):.3f}%)  <- false positives")
print("examples:", real_blocked[:15])
print(f"generated names blocked @T=0.8: {len(gen_blocked)} of {len(samples[0.8])} ({100*len(gen_blocked)/len(samples[0.8]):.2f}%)", gen_blocked[:10])
json.dump({"real_blocked": len(real_blocked), "real_total": len(tr), "gen_blocked_T0.8": len(gen_blocked),
           "gen_total": len(samples[0.8])}, open("out/filter_test.json", "w"), indent=2)

## Mascot with sd-turbo (diffusion)

In [ ]:
# Mascot with sd-turbo (diffusion model). Needs the GPU runtime. No API key needed.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers", "accelerate"], check=True)
from diffusers import AutoPipelineForText2Image
from PIL import Image
pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16").to("cuda")
prompt = ("cute friendly cartoon mascot, a smiling little wooden signpost beside a tiny Indian village hut and a mango tree, "
          "flat vector illustration, warm saffron and green colours, simple shapes, white background, no text, no letters")
imgs = []
for s in range(4):
    g = torch.Generator("cuda").manual_seed(s)
    im = pipe(prompt, num_inference_steps=2, guidance_scale=0.0, generator=g, height=512, width=512).images[0]
    im.save(f"out/mascot_{s}.png"); imgs.append(im)
grid = Image.new("RGB", (1024, 1024))
for i, im in enumerate(imgs): grid.paste(im, ((i % 2) * 512, (i // 2) * 512))
grid.save("out/mascot_grid.png"); shutil.copy("out/mascot_0.png", "out/mascot.png")
display(grid)
print("Pick your favourite (mascot_0..3.png). Default copied to out/mascot.png. Change the seed range or prompt and rerun if none is good.")

## Download

In [ ]:
!zip -j p18_bias_mascot.zip out/bias_chart.png out/bias_suffix_shares.csv out/memorisation.csv out/filter_test.json out/mascot*.png
from google.colab import files; files.download("p18_bias_mascot.zip")